# Text-side CLIP probe: ViT-L/14 (Colab, **CPU runtime is enough**)
Plan and B/32 result: `docs/TEXT_PROBE_PLAN.md`. Exploratory, not part of the preregistered study.

Runs CLIP's **text encoder only** on 960 prompts (no images are generated, loaded or shown):
1. **ViT-L/14**, the Stable Diffusion 1.x text encoder and D4's second model, with the same pinned weights as Phase 1 (sha256-checked).
2. **ViT-B/32 again** with Phase 1's pinned HF weights, as a cross-check: it should reproduce the local run (OpenAI's original .pt file) to within rounding.

Results are copied to Drive `child-detection-audit/results/text_probe/`. Needs the probe script pushed to GitHub first.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib, json, shutil, subprocess
DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/child-detection-audit')
REPO = pathlib.Path('/content/child-detection-skin-tone-audit')
URL = 'github.com/isaacmutuma/child-skintone-detection.git'
os.environ['CHILD_AUDIT_ENV'] = 'colab'
os.chdir('/content')
!rm -rf {REPO}
# Public repo: plain clone. Private repo: add a Colab secret GITHUB_TOKEN (key icon, left bar;
# a fine-grained token with read-only Contents access to this one repo).
if subprocess.run(['git', 'clone', '-q', f'https://{URL}', str(REPO)]).returncode != 0:
    from google.colab import userdata
    tok = userdata.get('GITHUB_TOKEN')
    subprocess.run(['git', 'clone', '-q', f'https://x-access-token:{tok}@{URL}', str(REPO)], check=True)
    subprocess.run(['git', '-C', str(REPO), 'remote', 'set-url', 'origin', f'https://{URL}'], check=True)
!git -C {REPO} log --oneline -1
os.chdir(REPO)
assert (REPO / 'scripts/text_probe.py').exists(), 'text_probe.py not on GitHub yet: commit and push first'
!pip -q install open_clip_torch==3.3.0
import open_clip, torch; print('open_clip', open_clip.__version__, '| torch', torch.__version__)

In [ ]:
# Keep the committed B/32 result (local run on OpenAI's .pt) for the cross-check
ref_path = REPO / 'results/text_probe/text_probe_ViT-B-32.json'
ref = json.loads(ref_path.read_text()) if ref_path.exists() else None
for arch in ['ViT-L-14', 'ViT-B-32']:
    print(f'===== {arch}')
    !python scripts/text_probe.py --arch {arch} --weights hf-pinned 2>&1 | grep -v -i warn

In [ ]:
out_dir = DRIVE_ROOT / 'results' / 'text_probe'; out_dir.mkdir(parents=True, exist_ok=True)
for arch in ['ViT-L-14', 'ViT-B-32']:
    d = json.loads((REPO / f'results/text_probe/text_probe_{arch}.json').read_text())
    assert d['git']['dirty'] is False, 'code changed after clone'
    name = f'text_probe_{arch}__colab_hfpinned.json'
    shutil.copy(REPO / f'results/text_probe/text_probe_{arch}.json', out_dir / name)
    bw = d['contrasts']['Black - White']['minor']
    print(f"{arch}: Black-White minor M1 {bw['m1_mean']:.4f} {bw['m1_ci']}  "
          f"({bw['m1_frac_of_child_adult_distance']:.2f} of distance), M2 {bw['m2_mean_years']:.2f} yrs; "
          f"decision: {'MATERIAL' if d['decision']['material_text_side_shift'] else 'not material'} "
          f"{d['decision']['criteria']} -> saved {name}")
# Cross-check: Phase 1 HF weights vs OpenAI original file should give the same B/32 numbers
if ref is not None:
    new = json.loads((REPO / 'results/text_probe/text_probe_ViT-B-32.json').read_text())
    diff = max(abs(ref['cells'][k][m] - new['cells'][k][m]) for k in ref['cells']
               for m in ['m1_mean_over_templates', 'm2_mean_over_templates'])
    print(f'B/32 cross-check, max |difference| over {len(ref["cells"])} cells: {diff:.2e}',
          '(OK: same weights)' if diff < 1e-3 else '(CHECK: weights or tokenizer differ)')